# Point-prediction emulator: what does a measured kNN-CDF predict?

Notebook 07's Fisher forecast answered "how tightly *could*
`(Omega_m, sigma_8)` be constrained" -- an achievable-precision bound,
not an actual prediction. This notebook builds the other half: given one
*measured* kNN-CDF summary, what point estimate of `(Omega_m, sigma_8)`
does it predict, trained on the LH suite and evaluated on genuinely
held-out data?

**This notebook's own results are not trustworthy until section 4's
shuffled-label null check clears** -- the same permutation-null
discipline this project applies everywhere else (the 1P trend test's
exact permutation p-value, `sensitivity_table`'s null floor, the CV
noise floor), now applied once to real data instead of many times to
synthetic fixtures. An emulator that looks predictive on shuffled targets
has a leakage or overfitting bug, not a real signal -- section 4 is a
required gate, not an optional extra, before reading section 3's numbers
as real.

Also not built here: a *rigorous* combined-tracer forecast (notebook 07's
naive Fisher combination assumed AGN/galaxy independence that real data
already showed is false, median correlation 0.69). This notebook's
combined-tracer model does *not* inherit that assumption -- concatenating
features lets a tree learn cross-tracer structure directly -- but that's
a different kind of "combined" than Fisher's, worth keeping straight when
comparing the two in section 5.


In [ ]:
import sys
sys.path.insert(0, "..")

import glob
import os
import time
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.params import load_params, align_to_params
from src.sensitivity import sensitivity_table, infer_layout
from src.emulator import (
    default_model, combined_features, cross_val_predict_emulator,
    cross_val_predict_many, train_full_model, prediction_metrics,
    null_control_metrics, feature_importance,
)

OUTPUT_DIR = config.OUTPUT_DIR
TARGETS = ["Omega_m", "sigma_8"]

## 1. Load the LH AGN and galaxy fixed-N runs

Same discover-and-align pattern as every comparison notebook here
(02/03/06/07): read each tracer's saved `.npz`, confirm it's really
fixed-N (`nbh` uniform), and align each to the full LH parameter table
via `params.align_to_params` -- never a positional join.


In [2]:
agn_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz"))
assert agn_candidates, f"No AGN fixed-N run found in {OUTPUT_DIR} -- run notebook 02 first."
agn_data = np.load(agn_candidates[-1], allow_pickle=True)
assert np.all(agn_data["nbh"] == agn_data["nbh"][0]), "AGN run is not fixed-N -- wrong file?"

galaxy_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/galaxy_knn_snap{config.SNAP}_M{config.GALAXY_MASS_CUT:.0e}_n*.npz"))
assert galaxy_candidates, f"No galaxy fixed-N run found in {OUTPUT_DIR} -- run notebook 06 first."
galaxy_data = np.load(galaxy_candidates[-1], allow_pickle=True)
assert np.all(galaxy_data["nbh"] == galaxy_data["nbh"][0]), "galaxy run is not fixed-N -- wrong file?"

theta_all = load_params(config.PARAMS_FILE)

agn_theta = align_to_params(agn_data["sim_ids"], theta_all)
agn_residuals = agn_data["summaries"] - agn_data["summaries"].mean(axis=0)
agn_n_k, agn_n_r = infer_layout(agn_data["kvals"], agn_data["rgrid"])

gal_theta = align_to_params(galaxy_data["sim_ids"], theta_all)
gal_residuals = galaxy_data["summaries"] - galaxy_data["summaries"].mean(axis=0)
gal_n_k, gal_n_r = infer_layout(galaxy_data["kvals"], galaxy_data["rgrid"])

print(f"AGN:    {len(agn_theta)} sims, {agn_residuals.shape[1]} bins")
print(f"Galaxy: {len(gal_theta)} sims, {gal_residuals.shape[1]} bins")


AGN:    950 sims, 150 bins
Galaxy: 950 sims, 150 bins


## 2. Three feature sets

AGN-only and galaxy-only are just each tracer's own residuals. Combined
uses `emulator.combined_features` (built on `complementarity.align_common_sims`
-- never a hand-rolled intersection), restricted to simulations both
runs retained.


In [3]:
common_ids, X_combined = combined_features(
    agn_data["sim_ids"], agn_residuals, galaxy_data["sim_ids"], gal_residuals,
)
combined_theta = align_to_params(common_ids, theta_all)

feature_sets = {
    "AGN": (agn_residuals, agn_theta),
    "galaxy": (gal_residuals, gal_theta),
    "combined": (X_combined, combined_theta),
}
for name, (X, theta) in feature_sets.items():
    print(f"{name:>8}: {X.shape[0]} sims, {X.shape[1]} features")


     AGN: 950 sims, 150 features
  galaxy: 950 sims, 150 features
combined: 929 sims, 300 features


## 3. Fit and evaluate: K-fold out-of-fold predictions

`cross_val_predict_emulator` gives every simulation a prediction from a
model that never saw it during training -- preferred over a single
train/test split at this sample size (every sim contributes to both
training and evaluation, rather than one split's luck deciding the
score). This is the direct empirical test of notebook 07's Fisher
forecast: does galaxy/combined actually beat AGN-only on `sigma_8`, as
Fisher predicted?

Calling that function once per feature set is only fold-parallel
*within* one feature set -- the next feature set doesn't start until the
current one's both targets finish, so on a machine with more cores than
one feature set's `n_splits` folds can use, that headroom sits idle
between feature sets, and there's no visibility into whether a feature
set is actually progressing or stuck. `cross_val_predict_many` instead
flattens every (feature set, target, fold) triple -- `3 feature sets x 2
targets x 10 folds = 60` independent 500-tree forest fits -- into ONE
`joblib.Parallel` dispatch, so every core is used across the whole
section at once, and `verbose=10` prints real progress ("Done k out of
60 | elapsed ... remaining ...") as fits complete. The cell below prints
the detected core count and the full fit count *before* dispatching, so
a persistently slow run and a genuinely large one are distinguishable
from the printed numbers alone, not by guessing after an interrupt.

If the printed pace shows fits completing correctly but the total is
still impractical for this session, the documented lever is fewer trees
for this comparison specifically -- e.g.
`model_factory=partial(default_model, n_estimators=200, n_jobs=1)` --
traded off against noisier held-out R^2/RMSE than the 500-tree default;
not applied here pre-emptively.

In [ ]:
print(f"detected cpu cores: {os.cpu_count()}")

n_splits = 10
n_trees = default_model().n_estimators
total_fits = len(feature_sets) * len(TARGETS) * n_splits
print(
    f"section 3 plan: {len(feature_sets)} feature sets x {len(TARGETS)} targets x "
    f"{n_splits} folds = {total_fits} independent random-forest fits ({n_trees} trees each)"
)

t0 = time.time()
predictions = cross_val_predict_many(
    feature_sets, TARGETS, model_factory=default_model,
    n_splits=n_splits, n_jobs=-1, verbose=10,
)
print(f"\nall {total_fits} fits done in {time.time() - t0:.1f}s total")

results = {}
rows = []
for name, (X, y, y_pred) in predictions.items():
    metrics = prediction_metrics(y, y_pred, target_names=TARGETS)
    metrics["feature_set"] = name
    results[name] = (X, y, y_pred)
    rows.append(metrics)

comparison = pd.concat(rows, ignore_index=True).set_index(["feature_set", "target"])
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, target in zip(axes, TARGETS):
    for name, (X, y, y_pred) in results.items():
        j = TARGETS.index(target)
        ax.scatter(y[:, j], y_pred[:, j], s=12, alpha=0.5, label=name)
    vals = np.concatenate([results[n][1][:, TARGETS.index(target)] for n in results])
    lo, hi = vals.min(), vals.max()
    ax.plot([lo, hi], [lo, hi], color="0.3", lw=1, ls="--")
    ax.set_xlabel(f"true {target}")
    ax.set_ylabel(f"predicted {target}")
    ax.legend(fontsize=8)
fig.tight_layout()


## 4. Required: shuffled-label null check on real data

Before trusting section 3's numbers: fit the identical pipeline with
`theta` shuffled (breaking the real summary<->parameter correspondence)
and confirm the *real* observed R^2/RMSE actually clears that null
distribution. Reduced `n_estimators`/`n_shuffles` here for tractability
(`n_shuffles x n_splits` model fits) -- same speed/rigor tradeoff
`sensitivity.py`'s tests make with `n_boot`/`n_null`.


In [ ]:
def _fast_for_null(random_state=42):
    from sklearn.ensemble import RandomForestRegressor
    # n_jobs=1: null_control_metrics parallelizes across folds itself
    # now (n_jobs=-1 below) -- see cross_val_predict_emulator's
    # docstring for why nesting n_jobs=-1 at both levels is avoided.
    return RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=1)

# Run the null check on whichever feature set looked best in section 3 --
# the others can be checked the same way by changing this name.
NULL_CHECK_FEATURE_SET = comparison.groupby("feature_set")["r2"].mean().idxmax()
print(f"Running the null check on: {NULL_CHECK_FEATURE_SET}")

X_check, y_check, y_pred_check = results[NULL_CHECK_FEATURE_SET]
obs = prediction_metrics(y_check, y_pred_check, target_names=TARGETS)
t0 = time.time()
null = null_control_metrics(
    X_check, y_check, target_names=TARGETS,
    model_factory=_fast_for_null, n_shuffles=25, n_splits=10, n_jobs=-1,
)
print(f"null check: {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
all_clear = True
for ax, target in zip(axes, TARGETS):
    null_r2 = null[null["target"] == target]["r2"]
    obs_r2 = obs.set_index("target").loc[target, "r2"]
    ax.hist(null_r2, bins=15, color="0.6", label="null (shuffled)")
    ax.axvline(obs_r2, color="#c0392b", lw=2, label="observed")
    ax.set_title(target)
    ax.set_xlabel("R^2")
    ax.legend(fontsize=8)

    clears = obs_r2 > np.percentile(null_r2, 95)
    all_clear = all_clear and clears
    verdict = "CLEARS" if clears else "DOES NOT CLEAR"
    print(f"{target}: observed R^2={obs_r2:.3f}, null 95th pct={np.percentile(null_r2, 95):.3f} -- {verdict} the null")

fig.tight_layout()

if not all_clear:
    print(
        "\n*** At least one target's observed R^2 did NOT clear its null distribution. ***\n"
        "*** Do not trust section 3's numbers for that target until this is understood. ***"
    )
else:
    print("\nBoth targets clear their null distributions -- section 3's numbers reflect real signal.")


## 5. Cross-check against notebook 07's Fisher forecast

Notebook 07's real Fisher numbers (from its saved outputs, not
hardcoded): `sigma(Omega_m)`/`sigma(sigma_8)` per feature set. Fisher's
diagonal-covariance sigma is already flagged optimistic (ignores real
bin-to-bin correlation) and assumes a locally linear response, so this
notebook's held-out RMSE should be the *same order of magnitude*, not
dramatically smaller -- a dramatically smaller RMSE here is a near-certain
leakage red flag, not evidence of a better-than-optimal model.


In [ ]:
from src.cv import per_bin_std
from src.fisher import (
    linear_response, fisher_matrix, combine_fisher, marginalized_covariance,
    sub_covariance, drop_uninformative_bins,
)

fisher_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/galaxy_cv_knn_snap{config.SNAP}_M{config.GALAXY_MASS_CUT:.0e}_n*.npz"))
agn_cv_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/cv_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz"))

if fisher_candidates and agn_cv_candidates:
    gal_cv_data = np.load(fisher_candidates[-1], allow_pickle=True)
    agn_cv_data = np.load(agn_cv_candidates[-1], allow_pickle=True)

    agn_cv_n_k, agn_cv_n_r = infer_layout(agn_cv_data["kvals"], agn_cv_data["rgrid"])
    gal_cv_n_k, gal_cv_n_r = infer_layout(gal_cv_data["kvals"], gal_cv_data["rgrid"])
    agn_cv_var = per_bin_std(agn_cv_data["summaries"], agn_cv_n_k, agn_cv_n_r).flatten() ** 2
    gal_cv_var = per_bin_std(gal_cv_data["summaries"], gal_cv_n_k, gal_cv_n_r).flatten() ** 2

    D_agn = linear_response(agn_residuals, agn_theta, params=config.ALL_PARAMS)
    D_gal = linear_response(gal_residuals, gal_theta, params=config.ALL_PARAMS)

    D_agn_kept, agn_cv_var_kept, _ = drop_uninformative_bins(D_agn, agn_cv_var)
    D_gal_kept, gal_cv_var_kept, _ = drop_uninformative_bins(D_gal, gal_cv_var)

    F_agn = fisher_matrix(D_agn_kept, agn_cv_var_kept)
    F_gal = fisher_matrix(D_gal_kept, gal_cv_var_kept)
    F_combined = combine_fisher(F_agn, F_gal)

    fisher_sigma = {}
    for name, F in [("AGN", F_agn), ("galaxy", F_gal), ("combined", F_combined)]:
        sub = sub_covariance(marginalized_covariance(F), config.ALL_PARAMS, TARGETS)
        fisher_sigma[name] = np.sqrt(np.diag(sub))

    print(f"{'feature_set':>10}  {'sigma(Omega_m)':>16}  {'sigma(sigma_8)':>16}  "
          f"{'RMSE(Omega_m)':>16}  {'RMSE(sigma_8)':>16}")
    for name in feature_sets:
        fs = fisher_sigma[name]
        rmse_om = comparison.loc[(name, 'Omega_m'), 'rmse']
        rmse_s8 = comparison.loc[(name, 'sigma_8'), 'rmse']
        print(f"{name:>10}  {fs[0]:16.4f}  {fs[1]:16.4f}  {rmse_om:16.4f}  {rmse_s8:16.4f}")
else:
    print(
        "Notebook 07's saved CV outputs weren't found in this OUTPUT_DIR -- "
        "run notebook 07 first for the Fisher cross-check, or skip this section."
    )


## 6. Feature importance: which scales does the model actually use?

Cross-checked qualitatively against `sensitivity_table`'s significant-bin
map -- if the random forest is weighting the same (k, r) region the
permutation-null-backed sensitivity analysis already flagged real, that's
a good sign the model learned real clustering structure, not an
artifact.


In [ ]:
agn_model = train_full_model(
    agn_residuals, agn_theta[TARGETS].values,
    model_factory=partial(default_model, n_jobs=-1),  # one-off fit, not looped -- full per-fit parallelism is the right call here
)
importance = feature_importance(agn_model, agn_n_k, agn_n_r)

fig, ax = plt.subplots(figsize=(7, 3.5))
for ki in range(agn_n_k):
    ax.plot(agn_data["rgrid"], importance[ki], label=f"k={agn_data['kvals'][ki]}")
ax.set_xscale("log")
ax.set_xlabel(r"$r\ [\mathrm{Mpc}/h]$")
ax.set_ylabel("RF feature importance")
ax.set_title("AGN model: which scales drive the (Omega_m, sigma_8) prediction")
ax.legend(fontsize=8)
fig.tight_layout()


## Reading this

- This is a **point predictor**, not a posterior and not the Fisher
  forecast -- its held-out RMSE is an empirical scatter, not a rigorous
  bound, and it returns one number per parameter, not a distribution.
- **K-fold out-of-fold, not a single train/test split**: every simulation
  gets exactly one held-out prediction, from a model that never saw it,
  using the whole LH suite for both training and evaluation.
- **Section 4's null check is not optional.** If a target didn't clear
  it, nothing else in this notebook should be trusted for that target
  until the cause (leakage, an alignment bug, too few effective degrees
  of freedom) is found and fixed.
- **Section 5's comparison is directional, not a hard bound.** Fisher's
  sigma is already an optimistic, linear-response-only number; this
  notebook's RMSE should land in the same neighborhood, not dramatically
  below it. Also: this notebook's "combined" feature set is not subject
  to Fisher's naive-independence caveat (`combine_fisher` assumed AGN and
  galaxies are independent, which they measurably aren't) -- a tree
  fit on concatenated features can exploit cross-tracer structure
  directly, so a real predictive edge for "combined" here would be a
  genuinely different, and stronger, finding than Fisher's optimistic
  upper bound.
